# HealthSpark-Claims-Analytics — Distributed Healthcare Claims Analytics & ML Pipeline

**End-to-end PySpark pipeline**: Synthetic Data Generation -> ETL -> Feature Engineering -> MLlib Model -> Evaluation

This notebook clones the **HealthSpark-Claims-Analytics** repository and runs the full pipeline using the project's source modules.

**Runtime**: Select **GPU** or **High-RAM** in Colab Pro for best performance.  
**Time**: ~8-12 minutes for the full pipeline including CrossValidator (81 model fits).

---

## Step 0 — Clone Repository & Install Dependencies

In [ ]:
import os

# Clone the HealthSpark-Claims-Analytics repository
REPO_URL = 'https://github.com/Deepak-Lingala/HealthSpark-Claims-Analytics.git'
PROJECT_DIR = '/content/HealthSpark-Claims-Analytics'

# Always reset to /content first to avoid stale directory errors
os.chdir('/content')

if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}

!git clone {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
!ls -la

In [ ]:
# Install dependencies — avoid touching numpy/pandas (Colab has compatible versions)
!pip install -q pyspark==3.5.3 faker>=33.1.0
print('Dependencies installed.')

In [ ]:
# Add project root to Python path so src/ modules are importable
import sys
sys.path.insert(0, PROJECT_DIR)

# Verify imports
from src.utils.spark_session import get_spark_session
from src.data_generation.generate_claims import main as generate_data
from src.pipeline.ingestion import ingest_all, CLAIMS_SCHEMA, PATIENTS_SCHEMA
from src.pipeline.transforms import run_all_transforms
from src.pipeline.feature_engineering import engineer_features
from src.pipeline.ml_pipeline import run_ml_pipeline
print('All project modules imported successfully.')

---
## Step 1 — Generate Synthetic Dataset (500K Claims)

In [ ]:
# Create data directories and generate synthetic healthcare claims
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/models', exist_ok=True)

generate_data()

# Verify generated files
!wc -l data/raw/claims.csv data/raw/patients.csv

---
## Step 2 — Initialize Spark & Run ETL Pipeline

In [ ]:
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# Initialize SparkSession using the project's factory
spark = get_spark_session(app_name='HealthSpark-Colab', shuffle_partitions=8)
print(f'Spark {spark.version} initialized with {spark.sparkContext.defaultParallelism} cores')

In [ ]:
# Run ingestion pipeline: CSV -> Schema validation -> QA checks -> Parquet
data_dir = os.path.join(PROJECT_DIR, 'data')
claims_df, patients_df = ingest_all(spark, data_dir)

---
## Step 3 — Run Transform Pipeline (Joins, Windows, Aggregations)

In [ ]:
# Transforms: broadcast joins, rolling windows, lag/lead, aggregations
# Runs both DataFrame API and Spark SQL approaches
enriched_df = run_all_transforms(spark, claims_df, patients_df)

---
## Step 4 — Feature Engineering (15+ ML Features)

In [ ]:
# Engineer features: risk scores, LOS ratio, provider stats, age buckets, cost features
features_df = engineer_features(spark, enriched_df)

---
## Step 5 — MLlib Pipeline + CrossValidator (27 combos x 3 folds = 81 fits)

```
StringIndexer -> OneHotEncoder -> VectorAssembler -> StandardScaler -> RandomForestClassifier
```

In [ ]:
# Train, tune, evaluate, and save the model
t0 = time.time()
run_ml_pipeline(spark, features_df, data_dir)
elapsed = time.time() - t0
print(f'\nTotal ML pipeline time: {elapsed:.0f}s ({elapsed/60:.1f} min)')

In [ ]:
# Load saved results
with open(os.path.join(data_dir, 'models', 'model_results.json')) as f:
    results = json.load(f)

metrics = results['metrics']
best_params = results['best_params']
feat_imp = results['feature_importances']

print('=' * 55)
print('  MODEL EVALUATION (Test Set)')
print('=' * 55)
for k, v in metrics.items():
    print(f'  {k:>15}: {v}')
print('-' * 55)
for k, v in best_params.items():
    print(f'  Best {k:>10}: {v}')
print('=' * 55)

---
## Step 6 — Visualizations

In [ ]:
# ============================================================
# 1. ROC CURVE
# ============================================================
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import roc_curve, auc as sk_auc, confusion_matrix

# Reload model and generate predictions for plotting
model = PipelineModel.load(os.path.join(data_dir, 'models', 'readmission_model'))

# Reload claims for prediction (need the feature-engineered version)
# Re-run feature engineering on a fresh load to avoid stale cache
claims_fresh = spark.read.option('header','true').option('dateFormat','yyyy-MM-dd').schema(CLAIMS_SCHEMA).csv(os.path.join(data_dir, 'raw', 'claims.csv'))

from src.pipeline.transforms import add_rolling_claim_cost, add_lag_lead_features
from src.pipeline.feature_engineering import (
    add_diagnosis_risk_score, add_los_ratio, add_provider_denial_rate,
    add_payer_approval_rate, add_age_bucket, add_cost_features, add_comorbidity_index
)

plot_df = claims_fresh
plot_df = add_rolling_claim_cost(plot_df)
plot_df = add_lag_lead_features(plot_df)
plot_df = add_diagnosis_risk_score(plot_df)
plot_df = add_los_ratio(plot_df)
plot_df = add_provider_denial_rate(plot_df)
plot_df = add_payer_approval_rate(plot_df)
plot_df = add_age_bucket(plot_df)
plot_df = add_cost_features(plot_df)
plot_df = add_comorbidity_index(plot_df)
plot_df = plot_df.fillna({
    'days_since_last_claim': -1, 'prev_denial_flag': 0, 'prev_claim_amount': 0.0,
    'rolling_cost_90d': 0.0, 'claim_count_90d': 0, 'claim_sequence': 1,
    'provider_denial_rate': 0.0, 'provider_claim_volume': 0,
    'payer_approval_rate': 0.85, 'patient_avg_claim': 0.0,
    'patient_total_claims': 1, 'cost_ratio_to_patient_avg': 1.0,
})
plot_df = plot_df.withColumn('readmission_30day', F.col('readmission_30day').cast('integer'))

# Use only test split
_, test_df = plot_df.randomSplit([0.8, 0.2], seed=42)
predictions = model.transform(test_df)

roc_data = (
    predictions
    .select('readmission_30day', vector_to_array('probability').alias('p'))
    .withColumn('prob_pos', F.col('p')[1])
    .select('readmission_30day', 'prob_pos')
    .toPandas()
)
fpr, tpr, _ = roc_curve(roc_data['readmission_30day'], roc_data['prob_pos'])
roc_auc = sk_auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='#1976D2', lw=2.5, label=f'ROC (AUC = {roc_auc:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#1976D2')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - 30-Day Readmission Prediction')
ax.legend(loc='lower right', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2. CONFUSION MATRIX
# ============================================================

pred_pd = predictions.select('readmission_30day', 'prediction').toPandas()
cm = confusion_matrix(pred_pd['readmission_30day'], pred_pd['prediction'])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
            xticklabels=['Not Readmitted','Readmitted'],
            yticklabels=['Not Readmitted','Readmitted'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - 30-Day Readmission')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3. FEATURE IMPORTANCE BAR CHART
# ============================================================

fi_df = pd.DataFrame(feat_imp[:15]).sort_values('importance')

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(fi_df['feature'], fi_df['importance'], color='#1976D2')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 15 Feature Importances - RandomForest Readmission Model')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4. DENIAL RATE BY PAYER
# ============================================================

denial_pd = (
    claims_fresh.groupBy('payer_type')
    .agg(F.avg('denial_flag').alias('denial_rate'), F.count('*').alias('n'))
    .orderBy(F.col('denial_rate').desc())
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(denial_pd['payer_type'], denial_pd['denial_rate'] * 100,
               color=sns.color_palette('YlOrRd', len(denial_pd)))
ax.set_xlabel('Denial Rate (%)')
ax.set_title('Claim Denial Rate by Payer Type')
for bar, rate in zip(bars, denial_pd['denial_rate']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{rate:.1%}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5. READMISSION RATE BY DIAGNOSIS
# ============================================================

readmit_pd = (
    claims_fresh.groupBy('diagnosis_code')
    .agg(F.avg('readmission_30day').alias('readmit_rate'), F.count('*').alias('n'))
    .where(F.col('n') >= 1000)
    .orderBy(F.col('readmit_rate').desc())
    .toPandas()
)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#d32f2f' if r > 0.20 else '#f57c00' if r > 0.15 else '#4caf50'
          for r in readmit_pd['readmit_rate']]
ax.barh(readmit_pd['diagnosis_code'], readmit_pd['readmit_rate'] * 100, color=colors)
ax.set_xlabel('30-Day Readmission Rate (%)')
ax.set_title('30-Day Readmission Rate by Diagnosis (ICD-10)')
ax.axvline(x=15, color='gray', linestyle='--', alpha=0.7, label='National avg ~15%')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6. COST DISTRIBUTION BY FACILITY TYPE
# ============================================================

cost_sample = claims_fresh.select('claim_amount','facility_type').sample(0.05, seed=42).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(cost_sample['claim_amount'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Claim Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Claim Amounts')
axes[0].set_xscale('log')
axes[0].axvline(cost_sample['claim_amount'].median(), color='red', linestyle='--',
                label=f'Median: ${cost_sample["claim_amount"].median():,.0f}')
axes[0].legend()

sns.boxplot(data=cost_sample, x='facility_type', y='claim_amount', ax=axes[1], showfliers=False)
axes[1].set_ylabel('Claim Amount ($)')
axes[1].set_title('Claim Cost by Facility Type')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7. AGE vs LENGTH OF STAY
# ============================================================

age_los = claims_fresh.select('age','length_of_stay','comorbidity_count').sample(0.02, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(age_los['age'], age_los['length_of_stay'],
                     c=age_los['comorbidity_count'], cmap='YlOrRd',
                     alpha=0.4, s=15, edgecolors='none')
plt.colorbar(scatter, label='Comorbidity Count')
ax.set_xlabel('Age')
ax.set_ylabel('Length of Stay (days)')
ax.set_title('Age vs Length of Stay (colored by comorbidity count)')
z = np.polyfit(age_los['age'], age_los['length_of_stay'], 1)
p = np.poly1d(z)
ax.plot(sorted(age_los['age'].unique()), p(sorted(age_los['age'].unique())),
        'b--', alpha=0.8, linewidth=2, label='Trend')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8. CORRELATION MATRIX
# ============================================================

corr_cols = ['readmission_30day','age','length_of_stay','claim_amount','paid_amount',
             'comorbidity_count','denial_flag']
corr_data = claims_fresh.select(corr_cols).sample(0.1, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = corr_data.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Matrix - Key Numeric Features')
plt.tight_layout()
plt.show()

---
## Final Summary

In [ ]:
print('=' * 60)
print('  HEALTHSPARK-CLAIMS-ANALYTICS - FULL PIPELINE RESULTS')
print('=' * 60)
print(f'\n  Dataset:')
print(f'    Claims:        500,000')
print(f'    Patients:      50,000')
print(f'\n  Model Performance:')
for k, v in metrics.items():
    print(f'    {k:>15}: {v}')
print(f'\n  Best Hyperparameters:')
for k, v in best_params.items():
    print(f'    {k:>15}: {v}')
print(f'\n  Top 5 Features:')
for i, f in enumerate(feat_imp[:5]):
    print(f'    {i+1}. {f["feature"]:<30} {f["importance"]:.4f}')
print('=' * 60)
print('\n  HealthSpark-Claims-Analytics: Distributed Healthcare Claims Analytics & ML Pipeline')
print(f'  Repository: https://github.com/Deepak-Lingala/HealthSpark-Claims-Analytics')

In [ ]:
spark.stop()
print('SparkSession stopped. Pipeline complete.')